Waiting on the data and processing it. Here will just print them.

Now let's read back the messages. The consumer receives raw bytes from Kafka. Instead of deserializing to a dict and then constructing a Ride manually, let's write a function that does both in one step:


Now we can pass ride_deserializer directly as the value_deserializer - Kafka calls it on every message, so message.value is already a Ride.

Connect to Kafka as a consumer. auto_offset_reset='earliest' means we start reading from the beginning of the topic (without this, new consumers default to latest and only see new messages). group_id identifies this consumer group - Kafka tracks how far each group has read, so restarting with the same group ID continues where it left off:

In [31]:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int  # epoch milliseconds

In [32]:
import json

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [33]:
# Test it with a sample JSON binary string (this is what Kafka delivers):
test_bytes = json.dumps({
    'PULocationID': 186,
    'DOLocationID': 79,
    'trip_distance': 1.72,
    'total_amount': 17.31,
    'tpep_pickup_datetime': 1730429702000
}).encode('utf-8')

ride_deserializer(test_bytes)

Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72, total_amount=17.31, tpep_pickup_datetime=1730429702000)

Now we can pass ride_deserializer directly as the value_deserializer - Kafka calls it on every message, so message.value is already a Ride.

Connect to Kafka as a consumer. auto_offset_reset='earliest' means we start reading from the beginning of the topic (without this, new consumers default to latest and only see new messages). group_id identifies this consumer group - Kafka tracks how far each group has read, so restarting with the same group ID continues where it left off:

In [37]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

In [38]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

Read messages and print them. Since value_deserializer returns a Ride, message.value is already a Ride object - no extra conversion needed:

In [39]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 20:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 20:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 20:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 20:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 20:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 20:13:25
Received: PU=142, DO=237, distance=2.28, amount=$24.94, pickup=2025-10-31 20:49:07
Received: PU=163, DO=238, distance=2.7, amount=$25.62, pickup=2025-10-31 20:07:19
Received: PU=138, DO=261, distance=12.87, amount=$86.14, pickup=2025-10-31 20:00:00
Received: PU=138, DO=37, distance=8.4, amount=$48.65, pickup=2025-10-31 20:18:50

... received 10 messages so far (stopping after 10 for demo)
